In [13]:
pip install torch torchvision pillow pillow-heif tqdm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
import os
from PIL import Image
import pillow_heif
from tqdm import tqdm

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for x, y in pbar:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

        avg_loss = total_loss / total
        avg_acc = correct / total
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")

    return total_loss / total, correct / total

In [17]:
@torch.no_grad() # 함수 전체에 torch.no_grad 적용
def run_eval(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc

In [18]:
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights)

train_dir = r"C:\Users\jy\260224-ViT\data\train"

In [19]:
# HEIC 열 수 있도록 등록
pillow_heif.register_heif_opener()

root = r"C:\Users\jy\260224-ViT\data\train"

def is_ftyp(path):
    try:
        with open(path, "rb") as f:
            head = f.read(32)
        return b"ftyp" in head
    except:
        return False


converted = 0
deleted = 0
failed = []

for dirpath, _, filenames in os.walk(root):
    for fn in filenames:
        if fn.lower().endswith((".jpg", ".jpeg")):
            path = os.path.join(dirpath, fn)

            if is_ftyp(path):  # 겉은 jpg인데 실제는 HEIC/HEIF 계열
                try:
                    img = Image.open(path).convert("RGB")

                    base, _ = os.path.splitext(path)
                    new_path = base + "_converted.jpg"

                    img.save(new_path, format="JPEG", quality=95)

                    # 저장 성공하면 원본 삭제
                    os.remove(path)

                    converted += 1
                    deleted += 1
                    print(f"[OK] Converted & Deleted: {path}")

                except Exception as e:
                    failed.append((path, str(e)))
                    print(f"[FAIL] {path} | {e}")

print("\n========== 결과 ==========")
# print("변환 성공:", converted)
# print("원본 삭제:", deleted)
# print("실패:", len(failed))

if failed:
    print("\n실패 파일 목록:")
    for p, e in failed:
        print(p, "|", e)


========== 결과 ==========


In [20]:
# 클래스 수 확인
dummy_ds = ImageFolder(train_dir)
num_classes = len(dummy_ds.classes)
print("클래스 수:", num_classes)

클래스 수: 4


In [21]:
# head 교체
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, num_classes)

# device 이동
model.to(device)

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [ ]:
# ViT(ImageNet pretrained) 기준 normalize
imgnt_mean = (0.485, 0.456, 0.406)
imgnt_std  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize((224,224)), # 3024 -> 224
    transforms.ToTensor(),
    transforms.Normalize(imgnt_mean, imgnt_std),
])

train_data = ImageFolder(train_dir, transform=train_tf)

train_loader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=(device == "cuda")
)

In [23]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
loss_fn = nn.CrossEntropyLoss()

In [24]:
# use_val = "val_loader" in globals()
# val_dir = "data/val"
# val_ds = ImageFolder(val_dir, transform=val_tf)
# val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=(device=="cuda"))

In [25]:
epochs = 10

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)

    msg = f"[Epoch {epoch:02d}] train loss={train_loss:.4f}, train acc={train_acc:.4f}"
    # if use_val:
    #     val_loss, val_acc = run_eval(model, val_loader, loss_fn, device)
    #     msg += f" | val loss={val_loss:.4f}, val acc={val_acc:.4f}"
    print(msg)

[Epoch 01] train loss=1.8960, train acc=0.3500


[Epoch 02] train loss=1.1331, train acc=0.5750


[Epoch 03] train loss=0.7490, train acc=0.7250


[Epoch 04] train loss=0.4204, train acc=0.8500


[Epoch 05] train loss=0.2044, train acc=0.9250


[Epoch 06] train loss=0.3755, train acc=0.8750


[Epoch 07] train loss=0.3179, train acc=0.8500


[Epoch 08] train loss=0.3816, train acc=0.8500


[Epoch 09] train loss=0.2354, train acc=0.9000


[Epoch 10] train loss=0.3058, train acc=0.9000
